# Pairs Trading — A Deep Dive

---

## What is Pairs Trading, and Why Does it Work?

Pairs trading is a **market-neutral** quantitative strategy. The core idea is deceptively simple: find two stocks that historically move together, and profit when they temporarily diverge.

Here's the intuition. Imagine two competing beverage giants — Coca-Cola (KO) and Pepsi (PEP). They sell similar products, compete for the same shelf space, are affected by the same input costs (sugar, aluminum, logistics), and are exposed to the same consumer spending trends. Their stock prices don't move in lockstep every single day — random news, earnings surprises, and short-term sentiment push them around independently. But over the long run, their *relationship* tends to be stable.

When Pepsi's stock suddenly outperforms Coke by an unusual amount — more than historical patterns would suggest — a pairs trader bets that the gap will close. They **short the outperformer** (PEP) and **go long the underperformer** (KO). If the relationship holds, both legs eventually move back toward each other and the trader profits on the convergence.

### Why is it called "market-neutral"?

Because you're simultaneously long and short with roughly equal dollar exposure. If the entire market sells off 5%, both KO and PEP will probably fall — but since you're long one and short the other, those market-wide moves largely cancel out. You're not betting on whether the market goes up or down. You're betting purely on the *relative* relationship between two stocks.

This is what makes pairs trading appealing: in theory, it should generate returns that are uncorrelated with broad market movements. In practice, this neutrality is imperfect, but it's much better than a purely directional bet.

### The statistical foundation: Cointegration

The formal mathematical concept behind pairs trading is **cointegration**. Two time series are cointegrated if they are both non-stationary individually (they wander randomly over time, like most stock prices), but some linear combination of them *is* stationary — meaning it has a stable, predictable long-run mean and doesn't drift off forever.

Think of it like two drunk people walking home together. Each person independently staggers around randomly — their individual paths are non-stationary. But they're linked: if one wanders too far from the other, they correct course. The *distance* between them is bounded and mean-reverting, even though each individual path is a random walk.

If two stocks are cointegrated, it means there's an economic or fundamental reason their relationship is stable — they're in the same industry, subject to the same macro forces, or one fundamentally depends on the other. The cointegration is a symptom of that underlying relationship.

### What can go wrong?

The biggest risk in pairs trading is **regime change** — the relationship between two stocks breaks down permanently. This could happen because:
- One company changes its business model (e.g., a beverage company acquires a tech startup)
- A regulatory change affects one company more than the other
- A merger or acquisition disrupts the pair
- One company goes bankrupt

When the relationship breaks, what looks like a "the spread will revert" trade becomes "this is a new normal" — and the position keeps losing. This is called a **convergence trade gone wrong**, and it's why stop-losses and ongoing monitoring of the spread's stationarity matter.

---

## Roadmap for this Notebook

1. **Fetch and visualize prices** — understand what we're working with
2. **Compute the spread** — learn what OLS regression is doing and why we use log-prices
3. **Test for stationarity** — ADF test: is our statistical assumption actually valid?
4. **Measure mean-reversion speed** — half-life: how long should we expect to hold a trade?
5. **Generate trading signals** — z-score thresholds and the state machine
6. **Try a second pair** — JPM / BAC to see the full approach generalize

In [ ]:
import sys
sys.path.insert(0, '../..')  # make src/ importable from notebooks/pairs/

from src.strategies.pairs.config import DEFAULT_PARAMS
from src.data.fetcher import fetch_pair
from src.strategies.pairs.spread import compute_hedge_ratio, compute_spread
from src.analytics.stationarity import adf_test, compute_half_life
from src.strategies.pairs.signals import compute_zscore, generate_signals
from src.strategies.pairs.viz import plot_prices, plot_spread, plot_zscore

---

## Section 1: Fetch Price Data

We start by pulling **2 years of daily adjusted closing prices** for KO and PEP using the `yfinance` library.

A few things worth understanding here:

**Why adjusted closing prices?**  
Raw closing prices are distorted by corporate actions like stock splits and dividends. When a company pays a dividend, its stock price drops by the dividend amount on the ex-dividend date — but you actually received that cash, so it's not a real loss. Adjusted prices account for splits and dividends so that the price series reflects the actual return you would have earned as a shareholder. For quantitative analysis, always use adjusted prices.

**Why 2 years?**  
This is a balance. Too little data (< 6 months) and your statistical tests won't have enough observations to be reliable. Too much data (> 5 years) and you risk including historical regimes that no longer reflect the current relationship between the companies. Two years is a reasonable default for swing trading — long enough to be statistically meaningful, short enough to stay relevant.

**Why daily bars?**  
For swing trading (holding positions for days to weeks), daily bars are the right granularity. Intraday data introduces a lot of microstructure noise — bid-ask spreads, order flow imbalances, lunch-hour effects — that isn't meaningful for multi-day trades. Daily closes are cleaner and far easier to work with.

In [2]:
TICKER_A = 'KO'
TICKER_B = 'PEP'
PERIOD   = DEFAULT_PARAMS['period']  # '2y'

df = fetch_pair(TICKER_A, TICKER_B, period=PERIOD)
print(f"Loaded {len(df)} trading days  ({df.index[0].date()} to {df.index[-1].date()})")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nLast 5 rows:")
print(df.tail())

Loaded 501 trading days  (2024-05-13 to 2026-05-12)

First 5 rows:
              close_a     close_b
date                             
2024-05-13  59.994476  168.333588
2024-05-14  59.541542  167.375153
2024-05-15  59.569847  166.993637
2024-05-16  59.749134  170.390076
2024-05-17  59.475487  169.533966

Last 5 rows:
              close_a     close_b
date                             
2026-05-06  79.230003  155.960007
2026-05-07  78.430000  156.289993
2026-05-08  78.419998  154.619995
2026-05-11  78.660004  149.410004
2026-05-12  79.535004  151.320007


In [3]:
fig = plot_prices(df, TICKER_A, TICKER_B)
fig.show()

**Reading the normalized price chart:**

Both prices are rebased to 100 at the start of the period. This lets us compare their relative performance on the same scale, regardless of the fact that KO and PEP have very different nominal price levels.

Notice how the two lines tend to move in the same direction most of the time — that's the cointegration at work. The interesting moments are when one line pulls noticeably ahead of or behind the other. Those divergences are what we'll be trading.

A key observation: even if one stock outperforms the other over the full 2-year period (i.e., the lines end at different levels), that doesn't necessarily break pairs trading. What matters is whether the *spread* is stationary — we'll test that formally in Section 3.

---

## Section 2: Computing the Spread

This is where the math starts getting interesting. We need to define "the spread" — a single number that captures the relative value of KO vs PEP at any given time.

### Why not just subtract prices?

The naive approach would be: `spread = price_KO - price_PEP`. But this has problems:

1. **Scale problem**: If KO trades at \$60 and PEP trades at \$180, subtracting them gives a spread dominated by PEP's price level, not by their relative movement.
2. **No dollar-neutrality**: If KO moves 1% and PEP moves 1%, a 1-share position in each isn't dollar-neutral — the dollar exposure to each stock depends on its price.

### Why log-prices?

We take the natural log of each price before working with them:

$$\log(P_{KO}) \text{ and } \log(P_{PEP})$$

This has several benefits:

- **Returns become additive**: $\log(P_t / P_{t-1}) = \log(P_t) - \log(P_{t-1})$ — this is the continuously compounded return, and differences in log-prices correspond to percentage moves, making everything scale-invariant.
- **Stabilizes variance**: Stock prices have roughly constant *percentage* volatility over time (not constant dollar volatility). Log-prices better reflect this.
- **Linearizes multiplicative relationships**: If KO and PEP move proportionally (both up 2%), their log-prices move by the same amount — exactly the stability we want.

### The Hedge Ratio via OLS Regression

We use **Ordinary Least Squares (OLS) regression** to find the hedge ratio β:

$$\log(P_{KO,t}) = \alpha + \beta \cdot \log(P_{PEP,t}) + \varepsilon_t$$

OLS finds the values of α and β that minimize the sum of squared residuals (the $\varepsilon_t$ terms). The slope β is our hedge ratio — it tells us how much $\log(P_{KO})$ tends to move for each unit move in $\log(P_{PEP})$.

**What does β mean practically?**  
β ≈ 1.0 means both stocks move proportionally (common for close competitors). β > 1 means KO is more volatile relative to PEP in log-space. β < 1 means PEP is more volatile.

To make the position as close to dollar-neutral as possible, you'd adjust your share counts based on β. But for this stage (signal generation only), β just helps us construct the most stationary spread possible.

### The Spread

Once we have β, the spread is:

$$\text{spread}_t = \log(P_{KO,t}) - \beta \cdot \log(P_{PEP,t})$$

This is the residual from the regression — how much KO "should" be worth given PEP's current price, versus where KO actually is. When this number deviates significantly from its historical average, we have a trading opportunity.

**A limitation worth knowing:** We're using the *full-sample* hedge ratio here — β is estimated using all 2 years of data at once. This is called an *in-sample* or *static* hedge ratio. In a real trading system, you'd re-estimate β periodically using only historical data available at the time (a *rolling* hedge ratio), to avoid look-ahead bias. For learning purposes, the static approach is cleaner to understand first.

In [4]:
beta = compute_hedge_ratio(df['close_a'], df['close_b'])
spread = compute_spread(df['close_a'], df['close_b'], beta)

print(f"Hedge ratio β  = {beta:.4f}")
print(f"Spread mean    = {spread.mean():.4f}")
print(f"Spread std dev = {spread.std():.4f}")
print(f"Spread min     = {spread.min():.4f}")
print(f"Spread max     = {spread.max():.4f}")

Hedge ratio β  = 0.0010
Spread mean    = 4.2054
Spread std dev = 0.0788
Spread min     = 4.0590
Spread max     = 4.3891


In [5]:
fig = plot_spread(spread, beta)
fig.show()

**Reading the spread chart:**

The spread should oscillate around a fairly stable mean (the dashed line) without any clear long-term upward or downward trend. If you see the spread steadily drifting in one direction over the full period, that's a warning sign — it may not be truly stationary, which would undermine the entire strategy.

The spread's scale is in log-space, so the values will be small (typically in the range of ±0.1 to ±0.5 for similar stocks). What matters isn't the absolute level — it's how far from the mean the spread is at any given time relative to its typical variation.

---

## Section 3: Statistical Tests

Before committing to a pairs trading strategy, we need to verify the statistical assumptions that justify it. There are two key tests:

1. **ADF Test** — Is the spread actually stationary? (validating the core assumption)
2. **Half-Life** — How fast does the spread mean-revert? (practical trading guidance)

---

### 3a. The Augmented Dickey-Fuller (ADF) Test

The ADF test is a formal hypothesis test for whether a time series has a **unit root** — which is statistician-speak for "does this series wander randomly without a tendency to return to a mean?"

**The hypotheses:**
- **H₀ (null hypothesis):** The spread has a unit root — it is NOT stationary. The pairs trading assumption is violated.
- **H₁ (alternative hypothesis):** The spread does NOT have a unit root — it IS stationary. The pairs trading assumption holds.

We want to *reject* H₀.

**The p-value:**
The p-value answers: "If the spread truly had a unit root, what is the probability of seeing test results this extreme by random chance?" A low p-value means it's unlikely we'd see these results if H₀ were true — so we reject H₀.

- **p-value < 0.05**: Reject H₀ at the 5% significance level → spread is stationary → pairs trading is justified ✓
- **p-value < 0.01**: Reject H₀ at the 1% level → very strong evidence of stationarity ✓✓
- **p-value ≥ 0.05**: Fail to reject H₀ → can't confirm stationarity → be very cautious ✗

**The ADF statistic:**
The test statistic itself is also reported. It's compared to critical values (at 1%, 5%, 10% significance). A more negative ADF statistic means stronger evidence against the unit root.

**Important caveat:** Passing the ADF test doesn't *guarantee* the spread will always revert. It means that based on historical data, the spread has behaved like a stationary process. The future relationship could break down — which is why ongoing monitoring matters in a live system.

In [6]:
adf = adf_test(spread)

print("=== Augmented Dickey-Fuller Test ===")
print(f"ADF Statistic : {adf['adf_stat']}")
print(f"p-value       : {adf['p_value']}")
print()
if adf['is_stationary']:
    print("✓ Spread is stationary (p < 0.05)")
    print("  The cointegration assumption holds — pairs trading is justified.")
else:
    print("✗ Cannot confirm stationarity (p >= 0.05)")
    print("  The spread may not be mean-reverting — proceed with caution.")

=== Augmented Dickey-Fuller Test ===
ADF Statistic : -1.0784
p-value       : 0.7236

✗ Cannot confirm stationarity (p >= 0.05)
  The spread may not be mean-reverting — proceed with caution.


---

### 3b. Half-Life of Mean Reversion

The ADF test tells us *whether* the spread reverts. The half-life tells us *how fast*.

**The math:**

We model the spread as a mean-reverting process. The simplest such model is an **Ornstein-Uhlenbeck (OU) process** in continuous time, which has a discrete-time approximation called an **AR(1) model**:

$$\Delta \text{spread}_t = \alpha \cdot \text{spread}_{t-1} + \varepsilon_t$$

Where:
- $\Delta \text{spread}_t = \text{spread}_t - \text{spread}_{t-1}$ is the change in the spread
- $\alpha$ is the speed of mean reversion (should be negative for mean reversion)
- $\varepsilon_t$ is random noise

The logic: when the spread is above its mean (positive), the change is negative (pushes it back down). When it's below the mean (negative), the change is positive. The magnitude of α tells us how aggressively it reverts.

We estimate α by regressing $\Delta \text{spread}$ on $\text{spread}_{t-1}$ (a simple linear regression with no intercept).

The **half-life** is then derived as:

$$\text{half-life} = \frac{-\ln(2)}{\alpha}$$

This is the expected number of days for the spread to move halfway back to its mean from any given deviation.

**Why does this matter for trading?**

| Half-life | Interpretation | Implication |
|---|---|---|
| < 5 days | Very fast reversion | More of an intraday/short-term play; daily bars may be too slow |
| 5–20 days | Fast reversion | Ideal for swing trading; positions typically close within a few weeks |
| 20–40 days | Moderate reversion | Still viable; needs larger z-score thresholds to ensure trades are worthwhile |
| 40–90 days | Slow reversion | Requires significant capital patience; large adverse moves possible |
| > 90 days | Very slow / unreliable | Questionable whether it's truly mean-reverting vs. just trending slowly |

**Practical guidance on the rolling window:**

When we compute z-scores, we use a rolling window (the number of days over which we calculate the mean and standard deviation). A good starting point is **2–3× the half-life**:

- Too short a window: the mean and std are noisy, generating lots of false signals
- Too long a window: the mean is sluggish, making the z-score slow to react to real divergences
- 2–3× the half-life is typically in the "Goldilocks" zone

In [7]:
hl = compute_half_life(spread)

print("=== Half-Life of Mean Reversion ===")
print(f"Half-life : {hl} trading days (~{hl/5:.1f} weeks)")
print()
print(f"Suggested rolling window: {int(hl * 2)}–{int(hl * 3)} days")
print(f"(Our default rolling window is {DEFAULT_PARAMS['rolling_window']} days)")
print()

if hl < 5:
    print("Note: Very short half-life — this pair may be better suited for shorter-term strategies.")
elif hl <= 40:
    print("Good: Half-life is in the swing-trading sweet spot.")
else:
    print("Note: Long half-life — trades may take weeks to resolve. Ensure stop-losses are set appropriately.")

=== Half-Life of Mean Reversion ===
Half-life : 109.7 trading days (~21.9 weeks)

Suggested rolling window: 219–329 days
(Our default rolling window is 60 days)

Note: Long half-life — trades may take weeks to resolve. Ensure stop-losses are set appropriately.


---

## Section 4: Generating Trading Signals

Now we translate the spread into actual trading signals. The key tool is the **rolling z-score**.

### The Z-Score

The z-score normalizes the spread relative to its recent history:

$$z_t = \frac{\text{spread}_t - \mu_{t,w}}{\sigma_{t,w}}$$

Where:
- $\mu_{t,w}$ = rolling mean of the spread over the last $w$ days
- $\sigma_{t,w}$ = rolling standard deviation of the spread over the last $w$ days

**Why z-score instead of raw spread?**

The raw spread level doesn't tell you much on its own. If the spread is at -0.05, is that a big deviation or a small one? The z-score answers that by expressing the deviation in units of standard deviations. A z-score of -2 means the spread is 2 standard deviations below its rolling mean — which is unusual enough (about 2.3% of observations under a normal distribution) to be worth trading.

**Why rolling (not full-sample)?**

We use a *rolling* window rather than the full-sample mean and std because:
1. It adapts to gradual shifts in the relationship between the stocks
2. It avoids look-ahead bias — at any point in time, we only use information available up to that date
3. Market regimes change; a rolling window lets the z-score reflect recent norms

---

### Entry and Exit Rules

Our signal logic uses three threshold levels:

| Parameter | Default | Meaning |
|---|---|---|
| `z_entry` | ±2.0 | Enter a trade when \|z\| exceeds this |
| `z_exit` | 0.5 | Exit (take profit) when \|z\| falls below this |
| `z_stop` | 3.0 | Stop-loss: exit when \|z\| exceeds this in the losing direction |

**Full signal rules:**

- **z < −2.0** (spread unusually low): **Long the spread** = buy KO, short PEP  
  *Why?* KO is cheap relative to PEP. We expect KO to rise and/or PEP to fall.

- **z > +2.0** (spread unusually high): **Short the spread** = short KO, buy PEP  
  *Why?* KO is expensive relative to PEP. We expect KO to fall and/or PEP to rise.

- **|z| < 0.5** (spread near mean): **Exit** — take profit, the spread reverted as expected.

- **|z| > 3.0** (spread moved further against us): **Stop out** — the relationship may be breaking down. Cut losses.

**The state machine:**

We don't just fire a signal every time z crosses a threshold — we track the current *position state*. While in a position, we ignore new entry signals (no doubling up or flipping). We only look for the exit or stop condition. Once flat, we look for new entries again.

This prevents rapid oscillation of signals near the threshold levels and ensures each trade has a clean lifecycle: enter → hold → exit/stop → repeat.

---

### Why ±2 standard deviations for entry?

Under a normal distribution, about 95% of observations fall within ±2 standard deviations. So a z-score beyond ±2 happens only about 5% of the time by random chance — it's unusual enough to be worth acting on.

In practice, the spread isn't perfectly normally distributed (it has fatter tails), but the ±2 rule remains a widely-used starting point in stat arb. You can tune this threshold:
- **Lower threshold (e.g., ±1.5)**: More frequent trades, but more false positives (trades that don't revert)
- **Higher threshold (e.g., ±2.5)**: Fewer trades, but stronger mean-reversion signals and potentially larger per-trade P&L

### Why exit at 0.5 instead of 0?

Waiting for a perfect return to z = 0 is greedy and often counterproductive:
- The spread bounces around near zero — waiting for exactly 0 means you'll miss exits and sometimes hold through an adverse move
- Exiting at |z| < 0.5 captures most of the available profit with less timing risk
- Transaction costs eat into profits on small residual moves near z = 0

In [8]:
params = DEFAULT_PARAMS

print("Strategy parameters:")
for k, v in params.items():
    print(f"  {k:<20} = {v}")

Strategy parameters:
  period               = 2y
  z_entry              = 2.0
  z_exit               = 0.5
  z_stop               = 3.0
  rolling_window       = 60


In [9]:
zscore = compute_zscore(spread, window=params['rolling_window'])
signals = generate_signals(
    zscore,
    z_entry=params['z_entry'],
    z_exit=params['z_exit'],
    z_stop=params['z_stop'],
)

print("=== Signal Summary ===")
print(f"Total bars in signal series : {len(signals)}")
print(f"Bars with active position   : {(signals['position'] != 0).sum()}")
print(f"Bars flat (no position)     : {(signals['position'] == 0).sum()}")
print()
print("Signal event counts:")
counts = signals['signal'].value_counts().dropna()
for event, count in counts.items():
    print(f"  {event:<20}: {count}")

=== Signal Summary ===
Total bars in signal series : 501
Bars with active position   : 232
Bars flat (no position)     : 269

Signal event counts:
  exit                : 8
  short_spread        : 5
  long_spread         : 3


In [10]:
fig = plot_zscore(signals, params['z_entry'], params['z_exit'], params['z_stop'])
fig.show()

**Reading the z-score chart:**

- **Purple line**: The rolling z-score. When it's near zero, the spread is at its historical average — no trade. When it's far from zero, there's a divergence worth trading.
- **Orange dotted lines** at ±2.0: Entry thresholds. Crossings trigger new trades.
- **Green shaded band** around zero: The exit zone (|z| < 0.5). When the z-score enters this band, open positions are closed.
- **Red dotted lines** at ±3.0: Stop-loss levels. If the spread keeps moving against us past ±3, we exit.
- **Blue up-triangles**: Long spread entries (buy KO, short PEP)
- **Orange down-triangles**: Short spread entries (short KO, buy PEP)
- **Green circles**: Exits (profitable close)
- **Red X's**: Stops (loss-limiting close)

Each triangle should be followed by either a circle (good) or an X (bad). If you see a triangle with no following marker before another triangle, look for a state machine bug.

**What a healthy z-score looks like:**  
It should oscillate reasonably symmetrically around zero, spending most of its time between ±2, with occasional spikes beyond that level. If the z-score trends strongly in one direction for an extended period, it suggests the spread is not truly stationary — time to recheck the ADF test or reconsider the pair.

---

## Section 5: Trying a Second Pair — JPM / BAC

JPMorgan Chase (JPM) and Bank of America (BAC) are two of the largest US commercial banks. They operate in essentially the same business — retail banking, investment banking, credit cards, mortgages — and are exposed to the same fundamental drivers:

- **Interest rates**: Banks profit from the spread between what they pay depositors and what they charge borrowers. Rising rates generally benefit both; falling rates compress margins for both.
- **Credit cycle**: Both profit in expansion (low defaults) and suffer in recession (high defaults).
- **Regulation**: Post-2008 rules like the Dodd-Frank Act apply similarly to both as systemically important financial institutions (SIFIs).

This shared fundamental exposure makes JPM/BAC a strong candidate for cointegration — temporary divergences in their stock prices should tend to revert as the underlying business drivers reassert themselves.

Let's run the full pipeline and compare it to KO/PEP.

In [11]:
df2 = fetch_pair('JPM', 'BAC', period=PERIOD)
beta2 = compute_hedge_ratio(df2['close_a'], df2['close_b'])
spread2 = compute_spread(df2['close_a'], df2['close_b'], beta2)
adf2 = adf_test(spread2)
hl2 = compute_half_life(spread2)

print("=== JPM / BAC ===")
print(f"Hedge ratio β  = {beta2:.4f}")
print(f"ADF p-value    = {adf2['p_value']}  {'✓ stationary' if adf2['is_stationary'] else '✗ not stationary'}")
print(f"Half-life      = {hl2} days (~{hl2/5:.1f} weeks)")
print()
print("=== KO / PEP (for comparison) ===")
print(f"Hedge ratio β  = {beta:.4f}")
print(f"ADF p-value    = {adf['p_value']}  {'✓ stationary' if adf['is_stationary'] else '✗ not stationary'}")
print(f"Half-life      = {hl} days (~{hl/5:.1f} weeks)")

=== JPM / BAC ===
Hedge ratio β  = 1.2425
ADF p-value    = 0.1366  ✗ not stationary
Half-life      = 30.2 days (~6.0 weeks)

=== KO / PEP (for comparison) ===
Hedge ratio β  = 0.0010
ADF p-value    = 0.7236  ✗ not stationary
Half-life      = 109.7 days (~21.9 weeks)


In [12]:
plot_prices(df2, 'JPM', 'BAC').show()

In [13]:
zscore2 = compute_zscore(spread2, window=params['rolling_window'])
signals2 = generate_signals(zscore2, z_entry=params['z_entry'], z_exit=params['z_exit'], z_stop=params['z_stop'])

print("=== JPM/BAC Signal Summary ===")
counts2 = signals2['signal'].value_counts().dropna()
for event, count in counts2.items():
    print(f"  {event:<20}: {count}")

=== JPM/BAC Signal Summary ===
  exit                : 9
  short_spread        : 5
  long_spread         : 4


In [14]:
plot_spread(spread2, beta2).show()
plot_zscore(signals2, params['z_entry'], params['z_exit'], params['z_stop']).show()

**What to compare between the two pairs:**

- **ADF p-values**: Which pair has stronger stationarity evidence?
- **Half-lives**: Which pair reverts faster? Does one require a different rolling window?
- **Number of trades**: More signals = more trades, but potentially more noise
- **Signal behavior**: Do entries tend to be followed by exits (good) or stops (bad)?

Financials pairs like JPM/BAC often show more volatile z-scores than consumer staples pairs like KO/PEP. Banks are subject to sudden credit events, regulatory announcements, and earnings surprises that cause sharp, sometimes sustained divergences. This can mean both more opportunity (bigger z-score excursions) and more risk (more stops).

Comparing two pairs side-by-side is a great way to develop intuition for what a "good" pair looks like — and why not every cointegrated pair is equally tradeable.

---

## Summary and What's Next

### What we built

You now have a complete pairs trading signal pipeline:

```
Raw daily prices (yfinance)
        ↓
OLS regression → Hedge ratio β
        ↓
Log-price spread = log(A) - β·log(B)
        ↓
ADF test → Is the spread stationary? (validates the assumption)
Half-life → How fast does it revert? (guides parameter choices)
        ↓
Rolling z-score → Where is the spread relative to recent history?
        ↓
State-machine signals → Long/Short/Exit/Stop
```

### Key concepts to remember

| Concept | One-line summary |
|---|---|
| Cointegration | Two random walks whose linear combination is stationary |
| Log-prices | Makes percentage moves additive; scale-invariant |
| OLS hedge ratio β | Minimizes residual variance; makes spread as stationary as possible |
| ADF test | Formally tests if the spread is stationary (p < 0.05 = stationary) |
| Half-life | Expected days to revert halfway; guides window and holding period |
| Z-score | Spread deviation in units of recent standard deviations |
| State machine | Tracks position to avoid overlapping trades |

### What's missing (and what's next)

This notebook produces *signals* — but not P&L. We don't yet know:
- How much money would these trades have made?
- What was the worst drawdown?
- What's the Sharpe ratio (risk-adjusted return)?
- How many of the trades were winners vs. losers?

That's the job of a **backtest engine** — the logical next step. The backtest takes the signals output and simulates actual trading, calculating equity curves, trade-by-trade P&L, and performance statistics.

Further down the road:
- **Rolling hedge ratio** — re-estimate β using only a trailing window (avoids look-ahead bias)
- **Automated pair discovery** — scan a universe of stocks for cointegrated pairs
- **Position sizing** — scale position by z-score magnitude or pair volatility
- **Portfolio management** — trade multiple pairs simultaneously with correlation awareness
- **Paper trading** — run signals live against real market data before risking capital